In [1]:
import pandas as pd

In [2]:
import pandas as pd
import os

# Automatically check multiple possible paths to locate your CSV file
possible_paths = [
    "data-analytics/data/raw/Crop_Production_Statistics.csv",
    "../data/raw/Crop_Production_Statistics.csv",
    "data/raw/Crop_Production_Statistics.csv",
    "./Crop_Production_Statistics.csv"
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Successfully loaded dataset from: {path}")
        break

if df is None:
    raise FileNotFoundError("❌ Could not find 'Crop_Production_Statistics.csv'. Please check your project folder structure.")

# Quick preview of loaded data
display(df.head())

✅ Successfully loaded dataset from: data/raw/Crop_Production_Statistics.csv


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75


In [3]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from all text/string columns
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

print("✅ Column names successfully cleaned:")
print(df.columns.tolist())

✅ Column names successfully cleaned:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']


In [4]:
# 1. Check missing values across all columns
print("🔍 Missing values per column:")
print(df.isnull().sum())

# 2. Drop rows where 'Crop' is missing (since yield analysis is impossible without a crop name)
initial_rows = len(df)
df = df.dropna(subset=['Crop'])
dropped_crops = initial_rows - len(df)
print(f"\n✨ Dropped {dropped_crops} rows with missing crop names.")
print(f"✨ Rows remaining: {len(df)}")

# 3. Inspect missing 'Production' values
missing_prod = df[df['Production'].isnull()]
print(f"\n📊 Total rows with missing production: {len(missing_prod)}")
display(missing_prod.head())

🔍 Missing values per column:
State            0
District         0
Crop             9
Crop_Year        0
Season           0
Area             0
Production    4948
Yield            0
dtype: int64

✨ Dropped 9 rows with missing crop names.
✨ Rows remaining: 345327

📊 Total rows with missing production: 4944


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
11,Andaman and Nicobar Island,NICOBARS,Arecanut,2005,Whole Year,795.67,NaN,0.0
270,Andaman and Nicobar Island,SOUTH ANDAMANS,Cowpea(Lobia),2016,Rabi,2.00,NaN,0.0
278,Andaman and Nicobar Island,NICOBARS,Dry chillies,2005,Whole Year,17.00,NaN,0.0
770,Andhra Pradesh,VIZIANAGARAM,Arecanut,2011,Whole Year,1.00,NaN,0.0
833,Andhra Pradesh,ANANTAPUR,Arhar/Tur,2016,Rabi,1.00,NaN,0.0


In [5]:
# 1. Check missing values across all columns
print("🔍 Missing values per column:")
print(df.isnull().sum())

# 2. Drop rows where 'Crop' is missing
df = df.dropna(subset=['Crop'])
print(f"\n✨ Rows remaining after dropping 9 missing crops: {len(df)}")

# 3. Inspect missing 'Production' values (4,948 rows)
missing_prod = df[df['Production'].isnull()]
print(f"📊 Total rows with missing production: {len(missing_prod)}")
display(missing_prod.head())

🔍 Missing values per column:
State            0
District         0
Crop             0
Crop_Year        0
Season           0
Area             0
Production    4944
Yield            0
dtype: int64

✨ Rows remaining after dropping 9 missing crops: 345327
📊 Total rows with missing production: 4944


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
11,Andaman and Nicobar Island,NICOBARS,Arecanut,2005,Whole Year,795.67,NaN,0.0
270,Andaman and Nicobar Island,SOUTH ANDAMANS,Cowpea(Lobia),2016,Rabi,2.00,NaN,0.0
278,Andaman and Nicobar Island,NICOBARS,Dry chillies,2005,Whole Year,17.00,NaN,0.0
770,Andhra Pradesh,VIZIANAGARAM,Arecanut,2011,Whole Year,1.00,NaN,0.0
833,Andhra Pradesh,ANANTAPUR,Arhar/Tur,2016,Rabi,1.00,NaN,0.0


In [6]:
# 1. Check rows where Production is missing but Area and Yield are present
missing_mask = df['Production'].isnull() & df['Area'].notnull() & df['Yield'].notnull()
print(f"📊 Rows where Production can be calculated using Area * Yield: {missing_mask.sum()}")

# 2. Fill missing Production using Area * Yield
df.loc[missing_mask, 'Production'] = df.loc[missing_mask, 'Area'] * df.loc[missing_mask, 'Yield']

# 3. Check remaining missing values after imputation
print("\n🔍 Remaining missing values per column:")
print(df.isnull().sum())

📊 Rows where Production can be calculated using Area * Yield: 4944

🔍 Remaining missing values per column:
State         0
District      0
Crop          0
Crop_Year     0
Season        0
Area          0
Production    0
Yield         0
dtype: int64


In [7]:
# 1. Check statistical summary of Yield to see min, max, median, and mean
print("📊 Yield Statistics:")
print(df['Yield'].describe())

# 2. Check the top 10 highest yields to see how extreme the outliers are
print("\n🚨 Top 10 Highest Yields:")
display(df.sort_values(by='Yield', ascending=False)[['State', 'District', 'Crop', 'Area', 'Production', 'Yield']].head(10))

📊 Yield Statistics:
count    345327.000000
mean         79.425204
std         916.690252
min           0.000000
25%           0.550000
50%           1.000000
75%           2.470000
max       43958.330000
Name: Yield, dtype: float64

🚨 Top 10 Highest Yields:


,State,District,Crop,Area,Production,Yield
24551,Assam,TINSUKIA,Coconut,72.0,3165000.0,43958.33
333525,West Bengal,BIRBHUM,Coconut,509.0,19749200.0,38800.00
258896,Tamil Nadu,ERODE,Coconut,10014.0,331800000.0,33133.61
24552,Assam,TINSUKIA,Coconut,71.0,2340000.0,32957.75
3122,Andhra Pradesh,PRAKASAM,Coconut,19.0,600000.0,31578.95
24548,Assam,TINSUKIA,Coconut,75.0,2257000.0,30093.33
3229,Andhra Pradesh,WEST GODAVARI,Coconut,23316.0,694164000.0,29772.00
24549,Assam,TINSUKIA,Coconut,76.0,2188000.0,28789.47
24303,Assam,KAMRUP,Coconut,2026.0,57405000.0,28334.16
24402,Assam,LAKHIMPUR,Coconut,192.0,5044000.0,26270.83


In [8]:
# Check unique units or group by crop to see average yields
print("📊 Average Yield by Top Crops:")
print(df.groupby('Crop')['Yield'].mean().sort_values(ascending=False).head(15))

📊 Average Yield by Top Crops:
Crop
Coconut         8833.247721
Sugarcane         55.592646
Banana            29.052143
Tapioca           18.905141
Onion             13.178732
Potato            13.149279
Jute               9.387052
Sweet potato       9.280385
Cashewnut          6.762511
Ginger             6.640230
Mesta              5.335766
Garlic             4.244649
Turmeric           3.540476
Maize              2.651930
Tobacco            2.648963
Name: Yield, dtype: float64


In [9]:
import numpy as np

# 1. Log Transformation to handle extreme skewness in Yield
df['Log_Yield'] = np.log1p(df['Yield'])

print("✅ Log transformation applied successfully to prevent outliers from breaking models!")
display(df[['Crop', 'Yield', 'Log_Yield']].head(10))

✅ Log transformation applied successfully to prevent outliers from breaking models!


,Crop,Yield,Log_Yield
0,Arecanut,1.40,0.875469
1,Arecanut,1.40,0.875469
2,Arecanut,0.74,0.553885
3,Arecanut,0.64,0.494696
4,Arecanut,0.75,0.559616
5,Arecanut,0.50,0.405465
6,Arecanut,1.59,0.951658
7,Arecanut,1.64,0.970779
8,Arecanut,1.66,0.978326
9,Arecanut,1.21,0.792993


In [11]:
# 1. Check unique state names using the correct column name 'State'
print("🗺️ Unique State Names before cleaning:")
print(sorted(df['State'].unique()))

🗺️ Unique State Names before cleaning:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'CHANDIGARH', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Laddak', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'THE DADRA AND NAGAR HAVELI', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [12]:
# 1. First, convert state names to Title Case to fix uppercase issues (like CHANDIGARH)
df['State'] = df['State'].str.title()

# 2. Define a mapping dictionary to fix duplicates/variations
state_mapping = {
    'The Dadra And Nagar Haveli': 'Dadra and Nagar Haveli',
    'Orissa': 'Odisha', # Just in case Orissa appears anywhere
    'Laddak': 'Ladakh'
}

# Apply the mapping
df['State'] = df['State'].replace(state_mapping)

# 3. Verify unique state names after cleaning
print("🗺️ Unique State Names after cleaning:")
print(sorted(df['State'].unique()))

🗺️ Unique State Names after cleaning:
['Andaman And Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli', 'Dadra and Nagar Haveli', 'Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [13]:
# 1. Map variations to a single standardized name format
state_mapping = {
    'Dadra And Nagar Haveli': 'Dadra and Nagar Haveli',
    'Andaman And Nicobar Island': 'Andaman and Nicobar Island',
    'Daman And Diu': 'Daman and Diu',
    'Jammu And Kashmir': 'Jammu and Kashmir'
}

df['State'] = df['State'].replace(state_mapping)

# 2. Final check for clean, unique state names
print("🗺️ Final Unique State Names:")
print(sorted(df['State'].unique()))
print(f"✨ Total unique states: {df['State'].nunique()}")

🗺️ Final Unique State Names:
['Andaman and Nicobar Island', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
✨ Total unique states: 36


In [14]:
# 1. Filter rows with zero production
zero_prod_df = df[df['Production'] == 0]
print(f"📊 Total rows with 0 Production: {len(zero_prod_df)}")

# 2. Check which crops have the most zero production rows
print("\nTop crops with 0 production:")
print(zero_prod_df['Crop'].value_counts().head(10))

📊 Total rows with 0 Production: 6409

Top crops with 0 production:
Crop
Sesamum               508
Moong(Green Gram)     426
Sannhamp              323
Urad                  285
Maize                 246
Castor seed           205
Cotton(lint)          191
Gram                  188
Moth                  177
Other  Rabi pulses    172
Name: count, dtype: int64


In [15]:
# 1. Check if rows with 0 production also have 0 area
zero_prod_with_area = df[(df['Production'] == 0) & (df['Area'] > 0)]
print(f"🌾 Rows with 0 Production but valid Area (Potential Crop Failures): {len(zero_prod_with_area)}")

# 2. Let's create a binary flag for 'Crop_Failure' so we don't lose this information
df['Crop_Failure'] = (df['Production'] == 0).astype(int)

print("✅ Added 'Crop_Failure' binary flag column successfully!")
display(df[['State', 'District', 'Crop', 'Area', 'Production', 'Crop_Failure']].head(5))

🌾 Rows with 0 Production but valid Area (Potential Crop Failures): 6409
✅ Added 'Crop_Failure' binary flag column successfully!


,State,District,Crop,Area,Production,Crop_Failure
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2439.6,3415.0,0
1,Andaman and Nicobar Island,NICOBARS,Arecanut,1626.4,2277.0,0
2,Andaman and Nicobar Island,NICOBARS,Arecanut,4147.0,3060.0,0
3,Andaman and Nicobar Island,NICOBARS,Arecanut,4147.0,2660.0,0
4,Andaman and Nicobar Island,NICOBARS,Arecanut,4153.0,3120.0,0
